## Chapter 11 Python应用：会计

## 爬取烘焙行业公司的财务数据（资产负债表），并将其保存为一个 pandas.DataFram

In [ ]:
#导入库
import requests
import json
import pandas as pd

#根据烘焙行业的接口url爬取数据，获取该行业下的全部公司信息
url = 'https://vip.stock.finance.sina.com.cn/quotes_service/api/json_v2.php/Market_Center.getHQNodeData?page=1&num=20&sort=symbol&asc=1&node=sw3_340802&symbol=&_s_r_a=setlen'
response = requests.get(url=url)
companydata=json.loads(response.text ) 

balance_value_rows = []#行业下各公司资产负债表

#循环行业公司信息后，按公司抓取报表数据
for company in companydata:
        symbol = company['symbol'] # 公司代码
        code = company['code'] #公司数字代码
        name = company['name'] # 公司名称
        # 资产负债表url
        balance_sheet_url = "https://quotes.sina.cn/cn/api/openapi.php/CompanyFinanceService.getFinanceReport2022?paperCode="+symbol+"&source=fzb&type=4&page=1&num=10"
       
        #抓取资产负债表数据并保存到表格中
        balance_data=[]#资产负债表数据信息
        balance_response = requests.get(balance_sheet_url)#发送资产负债表查询请求获得相应
        balance_data = json.loads(balance_response.text)['result']['data']['report_list']         
        for balance_date in balance_data.keys():#按报表日期循环
            balance_d = balance_data[balance_date]#按报表日期获取报表数据
            balance_value_row = {
                '报表日期':balance_date,
                '公司名称': name
            }            
            for balance_dd in balance_d['data']:#循环读取报表项目数据
                balance_dd_title = balance_dd['item_title'] #报表项目名称
                balance_dd_value = balance_dd['item_value'] #报表项目数据
                if balance_dd_value is not None and balance_dd_value != '':
                    balance_dd_value = float(balance_dd_value)
                else:
                    balance_dd_value = ""
                balance_value_row[balance_dd_title] = balance_dd_value
            balance_value_rows.append(balance_value_row)

# 数据保存到pd.DataFrame
_kd_spider_result = pd.DataFrame(balance_value_rows)
print(_kd_spider_result)


### 11.1 爬取表格型数据

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time, random

#### 11.1.1 Pandas.read_html函数

In [ ]:
url = 'http://quotes.money.163.com/data/caibao/xjllb_ALL.html?reportdate=20200331'
pd.read_html(url)[0].iloc[:3, :5]

In [83]:
def get_source_code(url, encoding = 'utf-8'):
    headers = {'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '\
    'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/79.0.3945.117 Safari/537.36'}
    response = requests.get(url, headers = headers, timeout = 5)
    response.encoding = encoding
    return response

In [20]:
pd.read_html(get_source_code(url).text)[0].iloc[:3, :5]

,序号,代码,名称,期末现金余额,经营现金流
0,1,605366,宏柏新材,1.85亿,"4,523.58万"
1,2,2993,奥海科技,3.33亿,1.31亿
2,3,605066,天正电气,2.30亿,"-9,079.60万"


#### 11.1.2 爬取网易财经财报数据

In [4]:
url = "http://quotes.money.163.com/data/caibao/yjgl_ALL.html?reportdate=20200331"
soup = BeautifulSoup(get_source_code(url).text, 'lxml')
tables = soup.find(name = 'div', attrs = {'class': 'fn_rp_submenu'})
print(tables)

<div class="fn_rp_submenu">
<a class="current" href="/data/caibao/yjgl_ALL.html?reportdate=20200331">业绩概览</a>
<a class="fn_rp_index" href="/data/caibao/zcfzb_ALL.html?reportdate=20200331">资产负债简表</a>
<a class="fn_rp_index" href="/data/caibao/lrb_ALL.html?reportdate=20200331">利润简表</a>
<a class="fn_rp_index" href="/data/caibao/xjllb_ALL.html?reportdate=20200331">现金流量简表</a>
<a class="fn_rp_index" href="/data/caibao/ylnl_ALL.html?reportdate=20200331">盈利能力</a>
<a class="fn_rp_index" href="/data/caibao/cznl_ALL.html?reportdate=20200331">偿债能力</a>
<a class="fn_rp_index" href="/data/caibao/ccnl_ALL.html?reportdate=20200331">成长能力</a>
<a class="fn_rp_index" href="/data/caibao/yynl_ALL.html?reportdate=20200331">营运能力</a>
</div>


In [5]:
table_names = [t.get_text() for t in tables.find_all(name = 'a')]
table_URLs = [t.attrs['href'] for t in tables.find_all(name = 'a')]

In [6]:
table_names

['业绩概览', '资产负债简表', '利润简表', '现金流量简表', '盈利能力', '偿债能力', '成长能力', '营运能力']

In [7]:
table_URLs

['/data/caibao/yjgl_ALL.html?reportdate=20200331',
 '/data/caibao/zcfzb_ALL.html?reportdate=20200331',
 '/data/caibao/lrb_ALL.html?reportdate=20200331',
 '/data/caibao/xjllb_ALL.html?reportdate=20200331',
 '/data/caibao/ylnl_ALL.html?reportdate=20200331',
 '/data/caibao/cznl_ALL.html?reportdate=20200331',
 '/data/caibao/ccnl_ALL.html?reportdate=20200331',
 '/data/caibao/yynl_ALL.html?reportdate=20200331']

In [22]:
writer = pd.ExcelWriter(r'..\examples\financial_statements.xlsx')
for i, name in enumerate(table_names):
    print("Getting table '{}' ...".format(name))
    url = 'http://quotes.money.163.com' + table_URLs[i]
    try:
        new_table = get_table(url)
        new_table.to_excel(writer, sheet_name = name, index = False)
        print("Successfully downloaded table '" + name + "'")
    except:
        print("Exception: Could not get table '" + name + "'")
        continue
writer.save()
writer.close()

Getting table '业绩概览' ...
Successfully got page 0
Exception: could not get page 1
Successfully got page 2
Successfully got page 3
Successfully got page 4
Successfully got page 5
Successfully got page 6
Successfully got page 7
Successfully got page 8
Successfully got page 9
Successfully got page 10
Successfully got page 11
Successfully got page 12
Successfully got page 13
Successfully got page 14
Successfully got page 15
Successfully got page 16
Successfully got page 17
Successfully got page 18
Successfully got page 19
Successfully got page 20
Successfully got page 21
Successfully got page 22
Successfully got page 23
Successfully got page 24
Successfully got page 25
Successfully got page 26
Successfully got page 27
Successfully got page 28
Successfully got page 29
Successfully got page 30
Successfully got page 31
Successfully got page 32
Successfully got page 33
Successfully got page 34
Successfully got page 35
Successfully got page 36
Successfully got page 37
Successfully got page 38
Su

Successfully got page 155
Successfully got page 156
Successfully got page 157
Successfully got page 158
Successfully got page 159
Successfully downloaded table '资产负债简表'
Getting table '利润简表' ...
Successfully got page 0
Successfully got page 1
Successfully got page 2
Successfully got page 3
Successfully got page 4
Successfully got page 5
Successfully got page 6
Successfully got page 7
Successfully got page 8
Successfully got page 9
Successfully got page 10
Successfully got page 11
Successfully got page 12
Successfully got page 13
Successfully got page 14
Successfully got page 15
Successfully got page 16
Successfully got page 17
Successfully got page 18
Successfully got page 19
Successfully got page 20
Successfully got page 21
Successfully got page 22
Successfully got page 23
Successfully got page 24
Successfully got page 25
Successfully got page 26
Successfully got page 27
Successfully got page 28
Successfully got page 29
Successfully got page 30
Successfully got page 31
Successfully got

Successfully got page 150
Successfully got page 151
Successfully got page 152
Exception: could not get page 153
Successfully got page 154
Successfully got page 155
Successfully got page 156
Successfully got page 157
Successfully got page 158
Successfully got page 159
Successfully downloaded table '现金流量简表'
Getting table '盈利能力' ...
Successfully got page 0
Exception: could not get page 1
Exception: could not get page 2
Successfully got page 3
Successfully got page 4
Successfully got page 5
Successfully got page 6
Exception: could not get page 7
Successfully got page 8
Successfully got page 9
Successfully got page 10
Successfully got page 11
Successfully got page 12
Successfully got page 13
Successfully got page 14
Successfully got page 15
Successfully got page 16
Successfully got page 17
Successfully got page 18
Successfully got page 19
Successfully got page 20
Successfully got page 21
Successfully got page 22
Successfully got page 23
Successfully got page 24
Successfully got page 25
Succ

Successfully got page 147
Successfully got page 148
Successfully got page 149
Successfully got page 150
Successfully got page 151
Successfully got page 152
Successfully got page 153
Successfully got page 154
Successfully got page 155
Successfully got page 156
Successfully got page 157
Successfully got page 158
Successfully got page 159
Successfully downloaded table '偿债能力'
Getting table '成长能力' ...
Successfully got page 0
Successfully got page 1
Successfully got page 2
Successfully got page 3
Successfully got page 4
Successfully got page 5
Successfully got page 6
Successfully got page 7
Successfully got page 8
Successfully got page 9
Exception: could not get page 10
Successfully got page 11
Successfully got page 12
Exception: could not get page 13
Successfully got page 14
Successfully got page 15
Successfully got page 16
Successfully got page 17
Exception: could not get page 18
Successfully got page 19
Successfully got page 20
Successfully got page 21
Exception: could not get page 22
Exc

Successfully got page 140
Successfully got page 141
Successfully got page 142
Successfully got page 143
Successfully got page 144
Successfully got page 145
Successfully got page 146
Successfully got page 147
Successfully got page 148
Successfully got page 149
Successfully got page 150
Successfully got page 151
Successfully got page 152
Successfully got page 153
Successfully got page 154
Successfully got page 155
Successfully got page 156
Successfully got page 157
Successfully got page 158
Successfully got page 159
Successfully downloaded table '营运能力'


In [15]:
def get_table(url):
    # 分页获取共160页数据
    df = pd.DataFrame()
    for p in range(160):
        page_url = url + '&sort=publishdate&order=desc&page=' + str(p)
        try:
            html = get_source_code(page_url).text
            new_page = pd.read_html(html, index_col = 0)[0]
            df = pd.concat([df, new_page], axis = 0, ignore_index = True)
            print("Successfully got page " + str(p))
        except:
            print("Exception: could not get page " + str(p))
        time.sleep(random.random() * 3)  # 随机休眠，防止访问过于频繁
    return df

### 11.2 财报数据的处理

#### 11.2.1 pdf文档解析

In [4]:
import pdfplumber
import pandas as pd

In [5]:
path = (r'examples/上市公司行业分类结果.pdf')
pdf = pdfplumber.open(path)
len(pdf.pages)

93

In [173]:
pdf.pages[2].extract_text().split('\n')[:5]

['门类名称及代码 行业大类代码 行业大类名称 上市公司代码 上市公司简称',
 '采矿业(B) 09 有色金属矿采选业 000506 中润资源',
 '000603 盛达资源',
 '000688 国城矿业',
 '000758 中色股份']

In [174]:
pdf.pages[0].extract_table()[:5]

[['门类名称及代码', '行业大类代码', '行业大类名称', '上市公司代码', '上市公司简称'],
 ['农、林、牧、渔业\n(A)', '01', '农业', '000998', '隆平高科'],
 [None, None, None, '002041', '登海种业'],
 [None, None, None, '002772', '众兴菌业'],
 [None, None, None, '300087', '荃银高科']]

In [4]:
pdf.close()

In [5]:
with pdfplumber.open(path) as pdf:
    df = pd.DataFrame()
    for p in pdf.pages: 
        page = p.extract_tables()[0]
        page = pd.DataFrame(page[1:], columns = page[0])
        df = pd.concat([df, page], axis = 0, ignore_index = True)

In [6]:
df.head()

,门类名称及代码,行业大类代码,行业大类名称,上市公司代码,上市公司简称
0,农、林、牧、渔业\n(A),01,农业,000998,隆平高科
1,None,None,None,002041,登海种业
2,None,None,None,002772,众兴菌业
3,None,None,None,300087,荃银高科
4,None,None,None,300189,神农科技


In [363]:
classified_companies = df.fillna(method = 'ffill')
classified_companies.head()

,门类名称及代码,行业大类代码,行业大类名称,上市公司代码,上市公司简称
0,农、林、牧、渔业\n(A),01,农业,000998,隆平高科
1,农、林、牧、渔业\n(A),01,农业,002041,登海种业
2,农、林、牧、渔业\n(A),01,农业,002772,众兴菌业
3,农、林、牧、渔业\n(A),01,农业,300087,荃银高科
4,农、林、牧、渔业\n(A),01,农业,300189,神农科技


In [364]:
classified_companies.set_index(['门类名称及代码', '行业大类名称', '上市公司代码'], inplace = True)
classified_companies

行业大类代码 上市公司简称
门类名称及代码       行业大类名称 上市公司代码              
农、林、牧、渔业\n(A) 农业     000998     01   隆平高科
                     002041     01   登海种业
                     002772     01   众兴菌业
                     300087     01   荃银高科
                     300189     01   神农科技
...                            ...    ...
综合(S)         综合     600624     90   复旦复华
                     600673     90    东阳光
                     600770     90   综艺股份
                     600784     90   鲁银投资
                     600805     90   悦达投资

[3878 rows x 2 columns]

In [365]:
classified_companies.reset_index(1, inplace = True)
classified_companies.head()

行业大类名称 行业大类代码 上市公司简称
门类名称及代码       上市公司代码                     
农、林、牧、渔业\n(A) 000998     农业     01   隆平高科
              002041     农业     01   登海种业
              002772     农业     01   众兴菌业
              300087     农业     01   荃银高科
              300189     农业     01   神农科技

#### 11.2.2 数据分组与聚合

In [199]:
def code_expand(code):
    # 在不足六位的股票代码的左侧补零
    s = str(code)
    while len(s) < 6:
        s = '0' + s  
    return s

company_property = pd.read_excel(r'..\examples\financial_statements.xlsx', 
                            sheet_name = '资产负债简表',  converters = {'代码': code_expand})
company_property.set_index(['代码'], inplace = True)
company_property.head()

,名称,总资产,货币资金,流动资产,总负债,流动负债,净资产,比上期,公告日期,详细
代码,,,,,,,,,,
605366,宏柏新材,11.60亿,2.00亿,6.13亿,3.20亿,2.91亿,8.39亿,--,2020-07-23,详细
002993,奥海科技,21.22亿,5.58亿,16.68亿,12.93亿,12.33亿,8.29亿,--,2020-07-23,详细
605066,天正电气,18.31亿,2.35亿,13.37亿,9.76亿,9.60亿,8.56亿,--,2020-07-21,详细
002995,天地在线,8.14亿,1.54亿,7.50亿,3.68亿,3.68亿,4.46亿,--,2020-07-20,详细
605100,华丰股份,14.60亿,2.96亿,8.45亿,6.13亿,4.64亿,8.47亿,--,2020-07-20,详细


In [366]:
company_info = pd.merge(classified_companies, company_property, 
         left_on = '上市公司代码', right_index = True)

In [369]:
company_info.iloc[: 5, 3: 7]

名称      总资产    货币资金    流动资产
门类名称及代码       上市公司代码                               
农、林、牧、渔业\n(A) 000998  隆平高科  151.55亿  13.95亿  68.17亿
              002041  登海种业   36.52亿   5.10亿  28.58亿
              002772  众兴菌业   53.66亿  12.26亿  21.21亿
              300087  荃银高科   19.77亿   4.47亿  15.40亿
              300189  神农科技   11.05亿   1.14亿   2.42亿

In [370]:
df = company_info[['名称', '流动资产', '流动负债']].replace('--', None)
df.dropna(inplace = True)

In [376]:
def str_to_num(num_str):
    num_str = num_str.replace(',', '')  # 去掉逗号
    if num_str[-2] == '万':  # x万亿
        return float(num_str[: -3]) * 10000
    if num_str[-1] == '万':  # x万
        return float(num_str[: -2]) * 0.0001
    else:  # 亿
        return float(num_str[: -2])

def most_liquid(df, row_num = 10):
    df['Liquidity_Ratio'] = df['流动资产'].apply(str_to_num) / df['流动负债'].apply(str_to_num)
    return df.sort_values(by = 'Liquidity_Ratio')[-row_num:]

In [372]:
df.groupby(level = 0, group_keys = False).apply(most_liquid, 3).head(6)

名称    流动资产       流动负债  Liquidity_Ratio
门类名称及代码          上市公司代码                                          
交通运输、仓储和\n邮政业(G) 601188  龙江交通  23.59亿      4.95亿         4.795918
                 002320  海峡股份  17.78亿      2.39亿         7.695652
                 603032  德新交运   4.12亿  3,516.06万        11.660978
住宿和餐饮业(H)        601007  金陵饭店  11.81亿      6.80亿         1.735294
                 002186   全聚德   8.69亿      4.65亿         1.869565
                 000524  岭南控股  23.54亿     11.32亿         2.079646

In [404]:
bins = [0, 1, 10, 100, 1000]
net_asset = company_info['净资产'].apply(str_to_num)
company_bins = pd.cut(net_asset, bins)
company_bins.head()

门类名称及代码        上市公司代码
农、林、牧、渔业\n(A)  000998    (10, 100]
               002041    (10, 100]
               002772    (10, 100]
               300087      (1, 10]
               300189    (10, 100]
Name: 净资产, dtype: category
Categories (4, interval[int64]): [(0, 1] < (1, 10] < (10, 100] < (100, 1000]]

In [405]:
pd.qcut(net_asset, 4)

门类名称及代码        上市公司代码
农、林、牧、渔业\n(A)  000998                (51.7, 27000.0]
               002041                  (22.85, 51.7]
               002772                  (22.85, 51.7]
               300087    (-150.30100000000002, 10.6]
               300189    (-150.30100000000002, 10.6]
                                    ...             
综合(S)          600624                  (10.6, 22.85]
               600673                (51.7, 27000.0]
               600770                  (22.85, 51.7]
               600784                  (10.6, 22.85]
               600805                (51.7, 27000.0]
Name: 净资产, Length: 3712, dtype: category
Categories (4, interval[float64]): [(-150.30100000000002, 10.6] < (10.6, 22.85] < (22.85, 51.7] < (51.7, 27000.0]]

In [406]:
net_asset.groupby(company_bins).count()

净资产
(0, 1]           33
(1, 10]         784
(10, 100]      2340
(100, 1000]     465
Name: 净资产, dtype: int64

In [409]:
labels = ['Small', 'Medium', 'Large', 'Too Big to Fail']
pd.cut(net_asset, bins, right = False, labels = labels).head()

门类名称及代码        上市公司代码
农、林、牧、渔业\n(A)  000998     Large
               002041     Large
               002772     Large
               300087    Medium
               300189     Large
Name: 净资产, dtype: category
Categories (4, object): [Small < Medium < Large < Too Big to Fail]

In [439]:
df = company_info.reset_index().iloc[11: 20, [5, 2, 6, 7, 8, 9]]
for i in range(2, 6):
    df[df.columns[i]] = df[df.columns[i]].apply(str_to_num)

In [438]:
df

,名称,行业大类名称,总资产,货币资金,流动资产,总负债
11,香梨股份,农业,2.9,1.00000,1.5,0.18586
12,新赛股份,农业,13.5,1.50000,5.7,7.10000
13,北大荒,农业,109.8,16.20000,61.4,41.80000
14,平潭发展,林业,43.5,2.90000,35.3,7.80000
15,ST云投,林业,33.8,0.82271,22.1,29.70000
16,福建金森,林业,18.9,2.90000,17.3,11.60000
17,ST景谷,林业,3.2,0.15688,2.6,2.90000
18,罗牛山,畜牧业,69.0,3.80000,17.2,26.60000
19,民和股份,畜牧业,36.0,9.70000,17.2,7.10000


In [443]:
pd.pivot_table(df, index = ['行业大类名称', '名称'])

总负债    总资产  流动资产      货币资金
行业大类名称 名称                                   
农业     北大荒   41.80000  109.8  61.4  16.20000
       新赛股份   7.10000   13.5   5.7   1.50000
       香梨股份   0.18586    2.9   1.5   1.00000
林业     ST云投  29.70000   33.8  22.1   0.82271
       ST景谷   2.90000    3.2   2.6   0.15688
       平潭发展   7.80000   43.5  35.3   2.90000
       福建金森  11.60000   18.9  17.3   2.90000
畜牧业    民和股份   7.10000   36.0  17.2   9.70000
       罗牛山   26.60000   69.0  17.2   3.80000

In [444]:
pd.pivot_table(df, index = ['行业大类名称'])

,总负债,总资产,流动资产,货币资金
行业大类名称,,,,
农业,16.361953,42.066667,22.866667,6.233333
林业,13.000000,24.850000,19.325000,1.694897
畜牧业,16.850000,52.500000,17.200000,6.750000


In [448]:
df.pivot_table(['总资产', '总负债'], index = '行业大类名称', aggfunc = max)

,总负债,总资产
行业大类名称,,
农业,41.8,109.8
林业,29.7,43.5
畜牧业,26.6,69.0
